# DS6021 Final Project
## Emmett Hannam, Jarrett Markman, Weston Williams, Jeffrey Zhang

### Data Cleaning

In [ ]:
# Import packages and libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor  
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet, LinearRegression
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import silhouette_score, mean_squared_error, accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from umap import UMAP
import umap.umap_ as umap

In [ ]:
# Build function to read in data based on input/output for a given week
def read_data(prefix, wk):
    # Set "file" equal to the inputted file path and week
    file = f"{prefix}_2023_w{wk}.csv"
    return pd.read_csv(file) # Return the file read
# Load all input data for a given week
def load_weekly_data(weeks=range(1, 19)): # For weeks 1-18
    # Create empty data frames
    input_data = pd.DataFrame()

    # Iterate through weeks
    for wk in weeks:
        # Set week as a string (e.g. 1 becomes "01")
        wk_str = f"{wk:02d}"

        # Read the data 
        input_df = read_data("train/input", wk_str)

        # Concatenate the data for each week
        input_data = pd.concat([input_data, input_df], ignore_index=True)

    return input_data
def load_supplementary():
    return pd.read_csv("train/supplementary_data.csv", low_memory=False)
input_data = load_weekly_data()

In [ ]:
# Clean tracking data
# Clean orientation and direction
input_data["o_clean"] = (-(input_data["o"] - 90)) % 360
input_data["dir_clean"] = (-(input_data["dir"] - 90)) % 360
# Set x on same scale based on play direction
input_data["x_clean"] = np.where(
      input_data["play_direction"] == "left",
      120 - input_data["x"],
      input_data[
          "x"
      ], 
  )
# y, s, a already clean
input_data["y_clean"] = input_data["y"]
input_data["s_clean"] = input_data["s"]
input_data["a_clean"] = input_data["a"]
# Clean orientation based on play direction
input_data["o_clean"] = np.where(
    input_data["play_direction"] == "left", 180 - input_data["o_clean"], input_data["o_clean"]
)
# Clean orientation, direction, vx, and vy
input_data["o_clean"] = (input_data["o_clean"] + 360) % 360 
input_data["dir_clean"] = (input_data["dir_clean"] + 360) % 360
input_data["dir_radians"] = np.radians(input_data["dir_clean"])
input_data["v_x"] = input_data["s_clean"] * np.cos(input_data["dir_radians"])
input_data["v_y"] = input_data["s_clean"] * np.sin(input_data["dir_radians"])
# Sort data based on nfl, game, play and frame id
input_data = input_data.sort_values(by=['nfl_id', 'game_id', 'play_id', 'frame_id'])
# Group by the specified columns and apply the shift within each group
input_data["prev_x"] = input_data.groupby(['nfl_id', 'game_id', 'play_id'])["x_clean"].shift(1)
input_data["prev_y"] = input_data.groupby(['nfl_id', 'game_id', 'play_id'])["y_clean"].shift(1)
# Remove the nans with no prev_x and prev_y
input_data = input_data.dropna(subset=["prev_x", "prev_y"])
# Remove features from input_data
df = input_data.drop(columns=['absolute_yardline_number', 'player_name', 'player_height', 'player_weight', 'player_birth_date', 'player_position', 'wk',
                             'x', 'y', 's', 'a', 'o', 'dir', 'play_direction', 'num_frames_output'])
# Sample 100000 rows
#df_sample = df.sample(n=100000, random_state=42)
# Output to csv
#df_sample.to_csv("input_data_clean.csv", index=False)

### EDA and Data Analysis

In [ ]:
# Look at all columns in df
df.columns

In [ ]:
# Summarize numeric features
numeric = ["x_clean", "y_clean", "s_clean", "a_clean", 
                "o_clean", "dir_clean", "v_x", "v_y"]
desc = df[numeric].describe().T
print(desc)

In [ ]:
# Plot speed and acceleration distributions
plt.figure(figsize=(14,6))

plt.subplot(1,2,1)
sns.histplot(df["s_clean"], bins=40, kde=True)
plt.title("Distribution of Player Speed")
plt.xlabel("Speed (yards/sec)")

plt.subplot(1,2,2)
sns.histplot(df["a_clean"], bins=40, kde=True, color="orange")
plt.title("Distribution of Player Acceleration")
plt.xlabel("Acceleration (yards/sec²)")

plt.tight_layout()
plt.show()

In [ ]:
# Speed and acceleration boxplots by position
plt.figure(figsize=(16,6))

plt.subplot(1,2,1)
sns.boxplot(data=df, x="player_position", y="s_clean")
plt.title("Speed by Position")

plt.subplot(1,2,2)
sns.boxplot(data=df, x="player_position", y="a_clean")
plt.title("Acceleration by Position")

plt.tight_layout()
plt.show()

In [ ]:
# Look at speed over time for all players in a sample game and play
sample_play = df[df["play_id"] == df["play_id"].iloc[0]]
sample_game = sample_play[sample_play["game_id"] == sample_play["game_id"].iloc[0]]

plt.figure(figsize=(10,5))
sns.lineplot(
    data=sample_play,
    x="frame_id",
    y="s_clean",
    hue="nfl_id",
    legend=False
)
plt.title("Speed Over Time for All Players During Sample game")
plt.xlabel("Frame ID")
plt.ylabel("Speed")
plt.show()

In [ ]:
# Look at velocity vectors for a specific play
play_id = df["play_id"].iloc[101]  

# specific play AND frame 1
play_frame = df[(df["play_id"] == play_id) & (df["frame_id"] == 1)].copy()

plt.figure(figsize=(12, 5))

plt.quiver(
    play_frame["x_clean"],
    play_frame["y_clean"],
    play_frame["v_x"],
    play_frame["v_y"],
    angles='xy',
    scale_units='xy',
    scale=1
)

plt.title(f"Velocity Vectors for Play {play_id} (Frame 1)")
plt.xlabel("X Position (yds)")
plt.ylabel("Y Position (yds)")
plt.xlim(0, 120)
plt.ylim(0, 53.3)

plt.show()

In [ ]:
# Look at unique yardline numbers over a game
gameID = df["game_id"].iloc[101]
playID = df["play_id"].iloc[101]
df2 = df[(df["game_id"] == gameID) & (df["player_name"] == 'Jared Goff')]
df2['absolute_yardline_number'].unique()

In [ ]:
# Look at the data for a random game_id
playIDRavens = df[(df['game_id'] == 2023123000)]
playIDRavens

### KMeans, PCA, PCR

In [ ]:
# Previewing data for later preparation 
df.head()

In [ ]:
# Look at shape
df.shape

In [ ]:
# Get value counts for player_side
df["player_side"].value_counts()

In [ ]:
# Look at columns in df
df.columns

#### PCA

In [ ]:
# Filter for only offensive data
offense = df[df['player_side'] == 'Offense'].copy()
# get pre snap data only
pre_snap = offense.loc[offense.groupby(['game_id', 'play_id', 'nfl_id'])['frame_id'].idxmin()]

feature_columns = ["x_clean", "y_clean", "s_clean", "a_clean", "o_clean", "dir_clean", "v_x", "v_y"]

X = pre_snap[feature_columns].values # Subset for predictors of interest above 

offense.head()

In [ ]:
pipe = Pipeline([ # pipeline definition, with only PCA to start of 
    ('scaler', StandardScaler()),
    ('pca', PCA())
])

pipe.fit(X) # fit the pipeline to data

pca_model = pipe.named_steps["pca"]
explained_var = pca_model.explained_variance_ratio_
cum_explained_var = np.cumsum(explained_var)
n_components = len(explained_var)

ev_df = pd.DataFrame({ # PCA metrics 
    "PC": np.arange(1, n_components + 1),
    "ExplainedVariance": explained_var,
    "CumulativeVariance": cum_explained_var
})


fig_scree = px.line( # plotly scree plot HERE
    ev_df, x="PC", y="ExplainedVariance",
    markers=True,
    title="Scree Plot: Proportion of Variance Explained"
)
fig_scree.show()

In [ ]:
ev_df

In [ ]:
X_pca_scores = pipe.transform(X)
pc_cols = [f"PC{i}" for i in range(1, n_components + 1)] 
# check transformed data and associated PC columns
scores_df = pd.DataFrame(X_pca_scores, columns=pc_cols)
scores_df

In [ ]:
# transforming data to check for optimal amount of clusters
silhouette = {}
inertias = {}

X_pca = pipe.transform(X) 

for k in range(2, 15): # check for optimal k-values
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(X_pca)

    silhouette[k] = silhouette_score(X_pca, kmeans.labels_)
    inertias[k] = kmeans.inertia_

In [ ]:
# Identify best k-value here based on silhouette score
optimal_k = max(silhouette, key = silhouette.get)

print("Best k based on Silhouette Score:", optimal_k)

#### KMeans

In [ ]:
# add kmeans in here, choosing k = 8, to try to get variability in the groups, before choosing the optimal number of clusters
# Redefining the FULL pipeline here with K-Means included

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),  
    ('kmeans', KMeans(n_clusters=8, n_init=10, random_state=42)) # 3,4 clusters, probbaly better, but want more to see more variabiltiy in different types of plays
])

pipe.fit(X)

pca = pipe.named_steps['pca']

labels = pipe.named_steps['kmeans'].labels_
pre_snap["cluster"] = labels

# stats, checking cluster summaries 

print("PCA Components:", pca.n_components_)
print("Explained variance (ratio):", pca.explained_variance_ratio_)
print("Cluster counts:\n", pre_snap["cluster"].value_counts())

explained_var = pca.explained_variance_ratio_
cum_explained_var = np.cumsum(explained_var)
n_components = len(explained_var)

df_pca = pd.DataFrame({
    "PC": np.arange(1, n_components + 1),
    "ExplainedVariance": explained_var,
    "CumulativeVariance": cum_explained_var
})

fig_scree = px.line(
    df_pca, x="PC", y="ExplainedVariance",
    markers=True,
    title="Scree Plot: Proportion of Variance Explained"
)
fig_scree.show()

In [ ]:
X_pca_scores = pipe.transform(X)
pc_cols = [f"PC{i}" for i in range(1, n_components + 1)]

scores_df = pd.DataFrame(X_pca_scores, columns=pc_cols) # PCA influenec by cluster
scores_df

In [ ]:
plt.figure(figsize=(8, 6)) # visualizing K-Means clusters in PCA space
sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=labels
)
plt.title("K-Means Clusters Visualized in PCA Space")
plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

In [ ]:
cluster_summary = pre_snap.groupby("cluster")[feature_columns].mean()
cluster_summary

In [ ]:
# this section was used to calculate silhouette scores for different k-values, but incorrect because it used original data, not PCA transformed data

#from sklearn.metrics import silhouette_score

#sil_scores = []
#K_values_sil = list(range(2, 11))

#for k in K_values_sil:
#    pipe.set_params(kmeans__n_clusters=k)
#    pipe.fit(X)

#    labels = pipe["kmeans"].labels_

#    sil = silhouette_score(X, labels)

#   sil_scores.append(sil)
#sil_scores

In [ ]:
# plotting silhouette scores for different k-values
lines = list(range(2, 11))
fig = px.line(
    x=lines,
    y=silhouette,
    markers= True,
    title="Silhouette Scores",
    labels={"x": "Number of K Clusters", "y": "Silhouette Scores"}
)

fig.show()

In [ ]:
# plotting elbow scores for different k-values

K_values = list(range(1, 11))
wcss = []

for k in K_values: # need within cluster sum of squares calculation HERE
    pipe.set_params(kmeans__n_clusters=k)
    pipe.fit(X)
    inertia = pipe["kmeans"].inertia_
    wcss.append(inertia)


fig = px.line( # plotting elbow plot for additional cluster evaluation
    x=K_values,
    y=wcss,
    markers=True,
    title="Elbow Plot",
    labels={"x": "Number of K Clusters", 
            "y": "WCSS"}
)

fig.show()

In [ ]:
wcss

#### KMeans, with $K$ = 4 as the optimal number of clusters

In [ ]:
pipe.set_params(kmeans__n_clusters=4) # setting K = 4
pipe.fit(X)

labels = pipe.named_steps['kmeans'].labels_
pre_snap["clusterk4"] = labels

X_pca = pipe.named_steps['pca'].transform(pipe.named_steps['scaler'].transform(X))

plt.figure(figsize=(10, 8)) # visualizing K-Means clusters in PCA space for K = 4
sns.scatterplot(x = X_pca[:, 0], y =X_pca[:, 1], hue=labels)
plt.title("K-Means for 4 Clusters in PCA Space")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.show()

In [ ]:
cluster3_summary = pre_snap.groupby("clusterk4")[feature_columns].mean()
print(cluster3_summary)

#### PCR

In [ ]:
features = ['prev_x', 'prev_y', 'o_clean', 'dir_radians', 's_clean', 'a_clean', 'v_x', 'v_y']
X = offense[features].values
y = offense['x_clean'].values

X_train, X_test, y_train, y_test = train_test_split( # Standard train-test split
    X, y, test_size=0.25, random_state=42
)

pcr_pipe = Pipeline([ # FULL PCR pipeline
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=4)),
    ("linreg", LinearRegression())
])

pcr_pipe.fit(X_train, y_train)# fitting the PCR model on the training data

In [ ]:
# Get train/test R^2 and predictions
train_r2 = pcr_pipe.score(X_train, y_train)
test_r2  = pcr_pipe.score(X_test, y_test)


y_train_pred = pcr_pipe.predict(X_train)
y_test_pred  = pcr_pipe.predict(X_test)


In [ ]:
# Get train/test RMSE
train_rmse = np.sqrt(np.mean((y_train - y_train_pred)**2))
test_rmse  = np.sqrt(np.mean((y_test - y_test_pred)**2))
# Print out model metrics
print(f"Using n_components = {4}")
print(f"Train R²  : {train_r2:.4f}")
print(f"Test  R²  : {test_r2:.4f}")
print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test  RMSE: {test_rmse:.4f}")

In [ ]:
# Get explained variance for PCA
pca_model = pcr_pipe.named_steps['pca']
print(f"\nExplained variance by 3 PCs: {pca_model.explained_variance_ratio_.sum():.4f}")
print(f"Single PC variance: {pca_model.explained_variance_ratio_}")

### Linear Regression

#### Goal 1: Measure the causal relationship between the tracking data and player coordinates

In measuring the causal relationship between the current player location, we chose to include:
- `prev_{loc}` (The previous x or y coordinate location)
- `o_clean` (Player orientation)
- `dir_radians` (The angle of player motion)
- `s_clean` (Speed in yards/second)
- `a_clean` (Acceleration in yards/second squared)
- `v_x` (Velocity along the x-axis)
- `v_y` (Velocity along the y-axis)

We can expect this predictors to collectively impact player movement, as they represent where they last were, where they are angled, and how and where they are moving. 

In [ ]:
# Use stats models to build a linear regression model for x_clean
# Set predictors and response
X_x = sm.add_constant(df[['prev_x', 'o_clean', 'dir_radians', 's_clean', 'a_clean', 'v_x', 'v_y']]) 
y_x = df[['x_clean']]
x_model = sm.OLS(y_x, X_x).fit()
print(x_model.summary())

In [ ]:
# Use stats models to build a linear regression model for y_clean
X_y = sm.add_constant(input_data[['prev_y', 'o_clean', 'dir_radians', 's_clean', 'a_clean', 'v_y', 'v_x']]) 
y_y = input_data[['y_clean']]
y_model = sm.OLS(y_y, X_y).fit() 
print(y_model.summary())

#### Test model assumptions (linearity, independence, vif, normality, heteroscedasticity)

In [ ]:
# Build a residuals/fitted plot to look at potential linearity and heteroscedasticity present
# Set plot parameters
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
# Set fitted values/residuals for x and y
fitted_vals_x = x_model.fittedvalues
residuals_x = x_model.resid
fitted_vals_y = y_model.fittedvalues
residuals_y = y_model.resid
# Build residuals/fitted plot for x and y coordinates
axes[0].scatter(fitted_vals_x, residuals_x, alpha=0.5)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel("Fitted values (x_model)")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs Fitted (x_model)")
axes[1].scatter(fitted_vals_y, residuals_y, alpha=0.5)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel("Fitted values (y_model)")
axes[1].set_title("Residuals vs Fitted (y_model)")
plt.tight_layout()
plt.show()

The residuals/fitted plot is a strong indication of *heteroscedasticity* present in the model, as the variance of errors is clearly not equal across fitted values. Additionally, the rhombus shape can be indicative of a non-linear presence within the model, and the fact that it's not truly capturing the relationship between the predictors and response. To combat these two issues, you can potentially perform various variable transformations among the response or predictor variables, which can include log terms, the inclusion of interaction terms polynomial terms, or removing variables from the model. 

In [ ]:
# VIF
def compute_vif(X):  
    # Drop intercept if present
    X_no_const = X.drop(columns=["const"], errors="ignore")
    
    vif_data = pd.DataFrame()
    vif_data["feature"] = X_no_const.columns
    vif_data["VIF"] = [
        variance_inflation_factor(X_no_const.values, i)
        for i in range(X_no_const.shape[1])
    ]
    return vif_data
print(compute_vif(X_x), compute_vif(X_y))

There doesn't appear to be any multicolinearity present within the model - though the VIF for `prev_y` in the y-coordinate model has a VIF > 5 which is a cause for concern. `prev_y` carries too much value in the model as it's representative of where the player last was, so it doesn't make sense to remove it. 

In [ ]:
# Test for normality of residuals using QQ plot for the x and y coordinate models
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sm.qqplot(x_model.resid, line='45', ax=axes[0])
axes[0].set_title("QQ Plot of Residuals (x_model)")
sm.qqplot(y_model.resid, line='45', ax=axes[1])
axes[1].set_title("QQ Plot of Residuals (y_model)")
plt.tight_layout()
plt.show()

In the qqplot, we can see that the data falls almost completely verticl and does not suggest that the sample distribution is any similar to the theoretical distribution. Given the distribution for theoretical quantiles hovers around 0, it's indicative that the data is extremely similar and clustered together as there isn't all too much different within the data. The normality assumption is violated.

Given the fact that the assumptions for linearity, heteroscedasticity, and normality appear to be violated (independence can be identified based on the *Durbin-Watson* value), we cannot interpret anything too much within this model. While features carry statistical significance and the model $R^2$'s are high, the assumptions being violated are indicative that the model results are unreliable, as they can lead to biased estimates and inaccurate predictions.

#### Goal 2: Attempt to predict player coordinates

While OLS can be effective in measuring the causal relationship between predictor variables with a given response, **Ridge** and **Lasso** regression can be effective in predicting a given response variable based on select predictors. **ElasticNet** is the best of both worlds, as it combines the coefficient shrinkage that *Ridge* performs, and also specific feature selection, like *Lasso*. Given the goal is to maximize predictive accuracy, we'll include the previous X and Y location in the model. 

In [ ]:
def build_model(target):
    if target == "x_clean":
        # Set numerical and categorical features
        nums = ['prev_x', 'o_clean', 'dir_radians', 'y_clean', 's_clean', 'a_clean', 'v_x', 'v_y']
        cats = ['player_side', 'player_role']
        X = input_data[nums + cats]
        y = input_data[['x_clean']]
    elif target == "y_clean":
        nums = ['prev_y', 'o_clean', 'dir_radians', 'x_clean', 's_clean', 'a_clean', 'v_x', 'v_y']
        cats = ['player_side', 'player_role']
        X = input_data[nums + cats]
        y = input_data[['y_clean']]
    return nums, cats, X, y
def fit_elastic_net(target, alphas=[0.1, 0.3, 0.5, 0.7, 0.8, 0.9], l1_ratio=0.5, test_size=0.2, random_state=22903):
    # Build the data
    nums, cats, X, y = build_model(target)
    y = y.values.ravel()  # flatten to 1D
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    # Preprocessing
    preprocess = ColumnTransformer([
        ("num", StandardScaler(), nums),
        ("cat", OneHotEncoder(), cats)
    ])
    
    best_alpha = None
    best_score = float("inf")
    best_model = None
    
    for alpha in alphas:
        model = Pipeline([
            ("preprocess", preprocess),
            ("elasticnet", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=1000, random_state=random_state))
        ])
        model.fit(X_train, y_train)
        preds = model.predict(X_test)  # evaluate on test set
        score = mean_squared_error(y_test, preds)
        
        if score < best_score:
            best_score = score
            best_alpha = alpha
            best_model = model
    
    print(f"Best alpha for {target}: {best_alpha}")
    print(f"Test RMSE: {np.sqrt(best_score):.4f}")
    return best_model, X_test, y_test

In [ ]:
enet_x_model, X_test_x, y_test_x = fit_elastic_net("x_clean")
enet_y_model, X_test_y, y_test_y = fit_elastic_net("y_clean")

In [ ]:
y_pred_x = enet_x_model.predict(X_test_x)
y_pred_y = enet_y_model.predict(X_test_y)

# Create a figure with 2 subplots, side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y_test_x, y_pred_x, alpha=0.5)
axes[0].plot(
    [y_test_x.min(), y_test_x.max()],
    [y_test_x.min(), y_test_x.max()],
    color="red", linestyle="--"
)
axes[0].set_title("X_clean: Actual vs Predicted (Test Set)")
axes[0].set_xlabel("Actual x_clean")
axes[0].set_ylabel("Predicted x_clean")
axes[0].grid(True)

axes[1].scatter(y_test_y, y_pred_y, alpha=0.5)
axes[1].plot(
    [y_test_y.min(), y_test_y.max()],
    [y_test_y.min(), y_test_y.max()],
    color="red", linestyle="--"
)
axes[1].set_title("Y_clean: Actual vs Predicted (Test Set)")
axes[1].set_xlabel("Actual y_clean")
axes[1].set_ylabel("Predicted y_clean")
axes[1].grid(True)

plt.tight_layout()
plt.show()

Based on the scatterplots above, we can see that the elastic net model was effective in predicting x locations, but struggled a little with predicting y locations. 

In [ ]:
def get_elasticnet_coefficients(model, nums, cats):
    """
    Extract coefficients from a fitted ElasticNet pipeline and return as a DataFrame.
    """
    # Preprocessor and model
    preprocessor = model.named_steps['preprocess']
    enet = model.named_steps['elasticnet']
    
    # Get feature names from preprocessing
    cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(cats)
    feature_names = np.concatenate([nums, cat_features])
    
    # Get coefficients
    coefs = enet.coef_
    
    # Combine into DataFrame
    coef_df = pd.DataFrame({
        'feature': feature_names,
        'coefficient': coefs
    }).sort_values(by='coefficient', key=abs, ascending=False)  # sort by magnitude
    
    return coef_df
# Get nums and cats for x_clean
nums_x, cats_x, _, _ = build_model("x_clean")
coef_x = get_elasticnet_coefficients(enet_x_model, nums_x, cats_x)
print("Elastic Net coefficients for x_clean:")
print(coef_x)
# Get nums and cats for y_clean
nums_y, cats_y, _, _ = build_model("y_clean")
coef_y = get_elasticnet_coefficients(enet_y_model, nums_y, cats_y)
print("\nElastic Net coefficients for y_clean:")
print(coef_y)

In looking at the coefficients for both the X and Y coordinate models, we can see the features being used to make these predictions. For the X-coordinate model, the previous X location as well as speed and accelerationhave a strong effect on current X, as well as the side of the ball and when a player is in defensive coverage. 

For the y-coordinate model the effects appear to come from the previous Y location, as well as the Y velocity, along with the orientation and the direction. 

In [ ]:
# Place functions built to run streamlit app
# Create build model function for building data for a linear regression model
def build_model(input_data, target):
    """
    Input: input_data, target

    Description: Build the data for a linear regression model

    Output: nums, cats, X, y
    """
    if target == "x_clean": # If the target is x_clean
        # Set numerical and categorical features
        nums = ['prev_x', 'o_clean', 'dir_radians', 'y_clean', 's_clean', 'a_clean', 'v_x', 'v_y']
        cats = ['player_side', 'player_role']
        X = input_data[nums + cats]
        y = input_data[['x_clean']]
    elif target == "y_clean": # If the target is y_clean
        nums = ['prev_y', 'o_clean', 'dir_radians', 'x_clean', 's_clean', 'a_clean', 'v_x', 'v_y']
        cats = ['player_side', 'player_role']
        X = input_data[nums + cats]
        y = input_data[['y_clean']]
    else: # If the target is not x_clean or y_clean
        raise ValueError("Invalid target. Must be 'x_clean' or 'y_clean'.")
    return nums, cats, X, y

# Function to fit an elastic net model based on parameters
def fit_elastic_net(input_data, target, alpha=0.1, l1_ratio=0.5, test_size=0.2):
    """
    Input: input_data, target, alpha, l1_ratio, test_size

    Description: Fit an elastic net model based on parameters

    Output: model, X_test, y_test, preds, nums, cats, rmse
    """
    # Build the data
    nums, cats, X, y = build_model(input_data, target)
    y = y.values.ravel()  # Flatten to 1D
    
    # Train/test split based on test size 
    X_train, X_test, y_train, y_test = train_test_split(
        # Split x, y, input test_size parameter
        X, y, test_size=test_size, random_state=22903
    )
    
    # Set model preprocessing with standard scaler and one hot encoder for numerical and categorical features
    preprocess = ColumnTransformer([
        ("num", StandardScaler(), nums),
        ("cat", OneHotEncoder(), cats)
    ])

    # Use preprocessing and elastic net to set the model pipeline
    model = Pipeline([
        ("preprocess", preprocess),
        # Build elastic net model based on given alpha and l1_ratio
        ("elasticnet", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=1000, random_state=22903))
    ])
    # Fit the model, make predictions, and get model RMSE
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    return model, X_test, y_test, preds, nums, cats, rmse

# Build function to get the elastic net coefficients for a given model
def get_elasticnet_coefficients(model, nums, cats):
    """
    Extract coefficients from a fitted ElasticNet pipeline and return as a DataFrame.
    """
    # Preprocessor and model
    # Get preprocessor and elastic net for model
    preprocessor = model.named_steps['preprocess']
    enet = model.named_steps['elasticnet']
    
    # Get feature names from preprocessing
    cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(cats)
    feature_names = np.concatenate([nums, cat_features])
    
    # Get coefficients
    coefs = enet.coef_
    
    # Combine into DataFrame
    coef_df = pd.DataFrame({
        'feature': feature_names,
        'coefficient': coefs
    }).sort_values(by='coefficient', key=abs, ascending=False)  # sort by magnitude
    
    return coef_df

### Logistic Regression

In [ ]:
"""
target_model.py

End-to-end pipeline to predict the targeted offensive player using
tracking-like data from input_2023_w01.csv.

Features include:
- Final-frame geometry
- Route shape (dx, dy, route_len)
- Distance to ball landing
- Defender separation & angle
- Number of defenders within 2 yards
- QB-relative orientation & distance (using player_position == 'QB')
"""

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer

# ---------------------------------------------------------------------
# Global paths (match your existing structure)
# ---------------------------------------------------------------------
BASE_DIR = ""#"./114239_nfl_competition_files_published_analytics_final"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
SUPPLEMENTARY_PATH = os.path.join(TRAIN_DIR, "supplementary_data.csv")

# ---------------------------------------------------------------------
# Global feature lists
# ---------------------------------------------------------------------
# Numeric tracking / geometry features (unchanged from before)
NUMERIC_BASE_FEATURE_COLS = [
    "x_clean", "y_clean",
    "x_start", "y_start",
    "dx", "dy", "route_len",
    "dist_to_ball",
    "s_clean", "a_clean",
    "s_start", "a_start",
    #"absolute_yardline_number",
    "num_frames_output",
    "nearest_defender",
    "defender_angle_cos",
    "num_defenders_within_2",
    "dist_to_qb",
    "ori_diff_to_qb_deg",
    "cos_ori_to_qb",
]

# High-value categorical features (from supplementary + tracking)
CATEGORICAL_FEATURE_COLS = [
    # from supplementary_data.csv
    # "route_of_targeted_receiver",
    # "pass_length",
    # "possession_team",
    "team_coverage_type",
    "team_coverage_man_zone",
    # "yardline_side",
    # "down",
    # "pass_result",
    # NEW: from tracking/offense vectors
    "player_position",
    "alignment_role",
]


def build_offense_vectors(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Given a raw tracking-like dataframe with columns like input_2023_w01.csv,
    return one row per offensive player per play with engineered features.

    Features:
      - Final frame geometry and start geometry
      - dist_to_ball, dx, dy, route_len
      - Defender separation & angle
      - num_defenders_within_2
      - QB-relative distance & orientation
      - NEW: alignment_role (outside/slot/backfield/inline with side)
      - player_position (carried through for categorical features)
    """
    df = df_raw.copy()

    # --- Final frame per player ---
    df["player_final_frame"] = df.groupby(
        ["game_id", "play_id", "nfl_id"]
    )["frame_id"].transform("max")
    final = df[df["frame_id"] == df["player_final_frame"]].copy()

    # --- First frame per player ---
    first = (
        df.sort_values("frame_id")
          .groupby(["game_id", "play_id", "nfl_id"])
          .first()
          .reset_index()
    )

    start_cols = ["game_id", "play_id", "nfl_id", "x_clean", "y_clean", "s_clean", "a_clean"]
    first_small = first[start_cols].rename(columns={
        "x_clean": "x_start",
        "y_clean": "y_start",
        "s_clean": "s_start",
        "a_clean": "a_start",
    })

    final = final.merge(
        first_small,
        on=["game_id", "play_id", "nfl_id"],
        how="left",
        validate="one_to_one",
    )

    # --- Geometry to ball & route shape ---
    final["dist_to_ball"] = np.sqrt(
        (final["x_clean"] - final["ball_land_x"]) ** 2
        + (final["y_clean"] - final["ball_land_y"]) ** 2
    )
    final["dx"] = final["x_clean"] - final["x_start"]
    final["dy"] = final["y_clean"] - final["y_start"]
    final["route_len"] = np.sqrt(final["dx"] ** 2 + final["dy"] ** 2)

    # --- Offense and Defense splits ---
    off = final[final["player_side"] == "Offense"].copy()
    def_final = final[final["player_side"] == "Defense"].copy()

    # ------------------------------------------------------------------
    # Defender-based features (nearest defender, angle, defenders within 2y)
    # ------------------------------------------------------------------
    if not def_final.empty and not off.empty:
        pairs = off.merge(
            def_final[["game_id", "play_id", "nfl_id", "x_clean", "y_clean"]],
            on=["game_id", "play_id"],
            suffixes=("_off", "_def"),
        )

        # Vector from offensive player to defender (final frame)
        pairs["vec_od_x"] = pairs["x_clean_def"] - pairs["x_clean_off"]
        pairs["vec_od_y"] = pairs["y_clean_def"] - pairs["y_clean_off"]

        # Distance to each defender
        pairs["dist_off_def"] = np.sqrt(
            pairs["vec_od_x"] ** 2 + pairs["vec_od_y"] ** 2
        )

        # Defenders within 2 yards
        pairs["within_2"] = (pairs["dist_off_def"] <= 2.0).astype(int)
        close_counts = (
            pairs
            .groupby(["game_id", "play_id", "nfl_id_off"])["within_2"]
            .sum()
            .reset_index()
            .rename(columns={
                "nfl_id_off": "nfl_id",
                "within_2": "num_defenders_within_2",
            })
        )

        # Angle between route movement (dx, dy) and vector to defender
        pairs["route_norm"] = np.sqrt(pairs["dx"] ** 2 + pairs["dy"] ** 2)
        pairs["def_vec_norm"] = np.sqrt(
            pairs["vec_od_x"] ** 2 + pairs["vec_od_y"] ** 2
        )

        eps = 1e-6
        dot = pairs["dx"] * pairs["vec_od_x"] + pairs["dy"] * pairs["vec_od_y"]
        denom = pairs["route_norm"] * pairs["def_vec_norm"] + eps
        pairs["defender_angle_cos"] = dot / denom  # in [-1, 1]

        # Nearest defender per offensive player
        pairs = pairs.sort_values(
            ["game_id", "play_id", "nfl_id_off", "dist_off_def"]
        )
        nearest = (
            pairs
            .groupby(["game_id", "play_id", "nfl_id_off"])
            .first()
            .reset_index()
            .rename(columns={
                "nfl_id_off": "nfl_id",
                "dist_off_def": "nearest_defender",
            })
        )

        # Merge nearest defender distance & angle
        off = off.merge(
            nearest[[
                "game_id",
                "play_id",
                "nfl_id",
                "nearest_defender",
                "defender_angle_cos",
            ]],
            on=["game_id", "play_id", "nfl_id"],
            how="left",
        )

        # Merge defenders-within-2 count
        off = off.merge(
            close_counts,
            on=["game_id", "play_id", "nfl_id"],
            how="left",
        )

        off["num_defenders_within_2"] = (
            off["num_defenders_within_2"]
            .fillna(0)
            .astype(int)
        )
    else:
        off["nearest_defender"] = np.nan
        off["defender_angle_cos"] = np.nan
        off["num_defenders_within_2"] = 0

    # ------------------------------------------------------------------
    # QB-based features using player_position == 'QB'
    # ------------------------------------------------------------------
    if "player_position" in final.columns:
        qb_final = final[
            (final["player_side"] == "Offense")
            & (final["player_position"] == "QB")
        ][["game_id", "play_id", "x_clean", "y_clean", "o_clean"]].copy()

        qb_final = qb_final.rename(columns={
            "x_clean": "qb_x_final",
            "y_clean": "qb_y_final",
            "o_clean": "qb_o_final",
        })
    else:
        qb_final = pd.DataFrame(
            columns=["game_id", "play_id", "qb_x_final", "qb_y_final", "qb_o_final"]
        )

    if not qb_final.empty:
        off = off.merge(
            qb_final,
            on=["game_id", "play_id"],
            how="left",
        )

        # Vector from player to QB
        off["vec_p_to_qb_x"] = off["qb_x_final"] - off["x_clean"]
        off["vec_p_to_qb_y"] = off["qb_y_final"] - off["y_clean"]

        # Angle from player to QB (degrees)
        off["angle_to_qb_deg"] = np.degrees(
            np.arctan2(off["vec_p_to_qb_y"], off["vec_p_to_qb_x"])
        )

        # Normalize orientations to [0, 360)
        off["o_norm"] = off["o_clean"] % 360
        off["angle_to_qb_norm"] = off["angle_to_qb_deg"] % 360

        # Smallest signed angle difference in [-180, 180]
        off["ori_diff_to_qb_deg"] = (
            (off["o_norm"] - off["angle_to_qb_norm"] + 180) % 360 - 180
        )

        # Cosine of that difference: 1 = facing QB, -1 = facing away
        off["cos_ori_to_qb"] = np.cos(
            np.deg2rad(off["ori_diff_to_qb_deg"])
        )

        # Distance to QB
        off["dist_to_qb"] = np.sqrt(
            off["vec_p_to_qb_x"] ** 2 + off["vec_p_to_qb_y"] ** 2
        )
    else:
        off["ori_diff_to_qb_deg"] = np.nan
        off["cos_ori_to_qb"] = np.nan
        off["dist_to_qb"] = np.nan

    # ------------------------------------------------------------------
    # Ensure player_position is present for offense rows
    # ------------------------------------------------------------------
    if "player_position" not in off.columns and "player_position" in df.columns:
        off = off.merge(
            df[["game_id", "play_id", "nfl_id", "player_position"]].drop_duplicates(),
            on=["game_id", "play_id", "nfl_id"],
            how="left",
        )

    # ------------------------------------------------------------------
    # NEW: alignment_role based on starting x/y and player_position
    # ------------------------------------------------------------------

    # Estimate field center and sideline bands from the data
    y_min = off["y_start"].min()
    y_max = off["y_start"].max()
    mid_field_y = 0.5 * (y_min + y_max)

    sideline_band = 0.15 * (y_max - y_min)
    left_sideline_cut = y_min + sideline_band
    right_sideline_cut = y_max - sideline_band

    # Backfield heuristic: behind LOS / absolute yardline by ~2 yards
    if "absolute_yardline_number" in off.columns:
        off["is_backfield"] = (
            off["x_start"] < (off["absolute_yardline_number"] - 2.0)
        ).astype(int)
    else:
        off["is_backfield"] = 0  # fallback

    # Side of field: right vs left relative to mid_field_y
    off["side_right"] = (off["y_start"] > mid_field_y).astype(int)

    # ---------------- classify_alignment: full definition ----------------
    def classify_alignment(row):
        """
        Classify an offensive player's alignment role based on:
          - player_position
          - is_backfield (behind LOS)
          - side_right (right vs left of field center)
          - y_start (how close to sideline vs middle)
        Returns a string like:
          - 'backfield_rb_left', 'backfield_rb_right'
          - 'inline_te_left', 'inline_te_right'
          - 'outside_left', 'outside_right'
          - 'slot_left', 'slot_right'
          - 'backfield_qb', 'backfield_other_left', etc.
        """
        pos = str(row.get("player_position", "UNK") or "UNK")
        y0 = row["y_start"]

        # Backfield roles (RB, FB, QB occasionally)
        if row["is_backfield"]:
            if pos in ("RB", "FB"):
                return "backfield_rb_right" if row["side_right"] else "backfield_rb_left"
            elif pos == "QB":
                return "backfield_qb"
            else:
                return "backfield_other_right" if row["side_right"] else "backfield_other_left"

        # Inline TE: TE on or near LOS
        if pos == "TE":
            return "inline_te_right" if row["side_right"] else "inline_te_left"

        # WR-like: outside vs slot based on distance from sideline
        # near sideline: "outside"; else: "slot"
        if y0 <= left_sideline_cut:
            return "outside_left"
        elif y0 >= right_sideline_cut:
            return "outside_right"
        else:
            return "slot_right" if row["side_right"] else "slot_left"
    # ---------------- end classify_alignment ----------------------------

    off["alignment_role"] = off.apply(classify_alignment, axis=1)

    return off


# ---------------------------------------------------------------------
# NEW helper: merge supplementary play-level features
# ---------------------------------------------------------------------
def merge_supplementary(off: pd.DataFrame) -> pd.DataFrame:
    """
    Merge play-level supplementary data (formations, coverage, route, etc.)
    into the offense-level per-player dataframe.

    Expects SUPPLEMENTARY_PATH to contain at least:
      - game_id, play_id
      - route_of_targeted_receiver
      - pass_length
      - possession_team
      - team_coverage_type
      - team_coverage_man_zone
      - yardline_side
      - down
      - pass_result
    """
    if not os.path.exists(SUPPLEMENTARY_PATH):
        raise FileNotFoundError(
            f"Supplementary file not found at {SUPPLEMENTARY_PATH}"
        )

    supp = pd.read_csv(SUPPLEMENTARY_PATH)

    # Ensure join keys are strings
    for df_ in (off, supp):
        df_["game_id"] = df_["game_id"].astype(str)
        df_["play_id"] = df_["play_id"].astype(str)

    # Limit supplementary columns to only what we need (plus keys)
    cols_to_keep = ["game_id", "play_id"] + CATEGORICAL_FEATURE_COLS
    cols_to_keep = [c for c in cols_to_keep if c in supp.columns]

    supp_small = supp[cols_to_keep].copy()

    off_merged = off.merge(
        supp_small,
        on=["game_id", "play_id"],
        how="left",
        validate="many_to_one",  # many players -> one play row
    )

    return off_merged

# ---------------------------------------------------------------------
# CHANGED: build_feature_matrix
# ---------------------------------------------------------------------
def build_feature_matrix(off_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build feature matrix X from offense-level dataframe.

    We do NOT preselect numeric / categorical columns here; we keep them
    in the DataFrame and let ColumnTransformer pick what it needs.

    We DO:
      - create the binary directional feature "play_dir_left"
    """
    X = off_df.copy()
    # encode play direction as a simple binary numeric feature
    X["play_dir_left"] = (off_df["play_direction"] == "left").astype(int)
    return X



# ---------------------------------------------------------------------
# CHANGED: train_and_evaluate – now uses ColumnTransformer with cat features
# ---------------------------------------------------------------------
def train_and_evaluate(df_raw: pd.DataFrame):
    """
    Build offense vectors, merge supplementary categorical features,
    train logistic regression model with numeric + categorical pipelines,
    and compute per-play metrics.
    """
    # 1) Build offense tracking features (unchanged logic)
    off = build_offense_vectors(df_raw)

    # 2) Merge supplementary play-level categorical info
    off = merge_supplementary(off)

    # 3) Label: targeted offensive player
    if "player_to_predict" not in off.columns:
        raise ValueError("Expected 'player_to_predict' column in data.")

    off["is_target"] = off["player_to_predict"].astype(int)

    # 4) Build feature matrix (adds play_dir_left)
    X = build_feature_matrix(off)
    y = off["is_target"].astype(int)

    # 5) Group by play for split (so we don't leak plays across splits)
    play_groups = (
        off[["game_id", "play_id"]]
        .astype(str)
        .agg("_".join, axis=1)
    )

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=play_groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    meta_test = off.iloc[test_idx].copy()

    # 6) Define which columns are numeric vs categorical for the transformer
    numeric_features = NUMERIC_BASE_FEATURE_COLS + ["play_dir_left"]
    categorical_features = CATEGORICAL_FEATURE_COLS

    # Filter for columns that actually exist (in case of missing ones)
    numeric_features = [c for c in numeric_features if c in X_train.columns]
    categorical_features = [c for c in categorical_features if c in X_train.columns]

    # 7) Preprocessing pipelines
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocess = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop",  # ignore any unused columns
    )

    # 8) Full model pipeline: preprocessing + logistic regression
    model = Pipeline(steps=[
        ("preprocess", preprocess),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            C=0.7,  # or even 0.3
        )),
    ])

    # 9) Fit model
    model.fit(X_train, y_train)

    # 10) Player-level probabilities (test only, for metrics)
    test_proba = model.predict_proba(X_test)[:, 1]
    meta_test["target_prob"] = test_proba

    # 11) Per-play evaluation (top-1 accuracy, mean prob, log-loss)
    meta_test["game_play"] = (
        meta_test["game_id"].astype(str) + "_" + meta_test["play_id"].astype(str)
    )

    play_groups = meta_test.groupby("game_play", group_keys=False)

    def per_play_stats(g: pd.DataFrame) -> pd.Series:
        # top-1 correctness (does argmax match true target?)
        best_idx = g["target_prob"].idxmax()
        top1_correct = int(g.loc[best_idx, "is_target"])

        # probability assigned to true target
        true_prob = g.loc[g["is_target"] == 1, "target_prob"].iloc[0]

        # positive log-loss: -log(p_true)
        pos_log_loss = -np.log(max(true_prob, 1e-15))

        return pd.Series({
            "top1_correct": top1_correct,
            "true_target_prob": true_prob,
            "pos_log_loss": pos_log_loss,
        })

    play_stats = play_groups.apply(per_play_stats)

    top1_accuracy = play_stats["top1_correct"].mean()
    mean_true_prob = play_stats["true_target_prob"].mean()
    mean_pos_log_loss = play_stats["pos_log_loss"].mean()

    print(f"Per-play top-1 accuracy: {top1_accuracy:.6f}")
    print(f"Mean true target probability: {mean_true_prob:.6f}")
    print(f"Mean positive log-loss: {mean_pos_log_loss:.6f}")

    return model, off, play_stats


# ---------------------------------------------------------------------
# Unchanged function: predict_targets_for_week (if you still use it)
# ---------------------------------------------------------------------
def predict_targets_for_week(model, df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    [UNCHANGED or OPTIONAL: paste your previous implementation here,
    but you likely want to:
      - call build_offense_vectors(df_raw)
      - call merge_supplementary(off)
      - call build_feature_matrix(off)
      - then use model.predict_proba on X_new
    ]
    """
    raise NotImplementedError("Optionally update this based on your previous version.")


# ---------------------------------------------------------------------
# CHANGED: evaluate_week_with_model – also merges supplementary
# ---------------------------------------------------------------------
def evaluate_week_with_model(model, csv_path: str):
    """
    Load a week file, run the existing trained model on it,
    and if labels (player_to_predict) exist, compute per-play metrics.
    Otherwise, just return predictions and NaN metrics.

    Returns
    -------
    off : pd.DataFrame
        Per-offensive-player predictions for that week.
    metrics : dict
        Dictionary with keys:
            - top1_accuracy
            - mean_true_prob
            - mean_pos_log_loss
        Values are floats (or NaN if labels missing).
    """
    print(f"\n=== Evaluating file: {csv_path} ===")
    df_week = pd.read_csv(csv_path)

    off = build_offense_vectors(df_week)
    off = merge_supplementary(off)
    has_labels = "player_to_predict" in off.columns

    X_week = build_feature_matrix(off)

    # Same preprocessing as training: imputer + scaler are inside the model pipeline
    proba = model.predict_proba(X_week)[:, 1]
    off["target_prob"] = proba

    # Build a prediction flag per play
    off["game_play"] = (
        off["game_id"].astype(str) + "_" + off["play_id"].astype(str)
    )
    off["predicted_target"] = False
    for gp, idxs in off.groupby("game_play").groups.items():
        g = off.loc[idxs]
        best_idx = g["target_prob"].idxmax()
        off.loc[best_idx, "predicted_target"] = True

    metrics = {
        "top1_accuracy": np.nan,
        "mean_true_prob": np.nan,
        "mean_pos_log_loss": np.nan,
    }

    if has_labels:
        off["is_target"] = off["player_to_predict"].astype(int)

        meta_test = off.copy()
        meta_test["game_play"] = (
            meta_test["game_id"].astype(str)
            + "_"
            + meta_test["play_id"].astype(str)
        )
        play_groups = meta_test.groupby("game_play", group_keys=False)

        def per_play_stats(g: pd.DataFrame) -> pd.Series:
            best_idx = g["target_prob"].idxmax()
            top1_correct = int(g.loc[best_idx, "is_target"])
            true_prob = g.loc[g["is_target"] == 1, "target_prob"].iloc[0]
            pos_log_loss = -np.log(max(true_prob, 1e-15))
            return pd.Series({
                "top1_correct": top1_correct,
                "true_target_prob": true_prob,
                "pos_log_loss": pos_log_loss,
            })

        play_stats = play_groups.apply(per_play_stats)

        top1_accuracy = play_stats["top1_correct"].mean()
        mean_true_prob = play_stats["true_target_prob"].mean()
        mean_pos_log_loss = play_stats["pos_log_loss"].mean()

        print(f"Per-play top-1 accuracy: {top1_accuracy:.6f}")
        print(f"Mean true target probability: {mean_true_prob:.6f}")
        print(f"Mean positive log-loss: {mean_pos_log_loss:.6f}")

        metrics = {
            "top1_accuracy": float(top1_accuracy),
            "mean_true_prob": float(mean_true_prob),
            "mean_pos_log_loss": float(mean_pos_log_loss),
        }
    else:
        print("No 'player_to_predict' column found; returning predictions only.")

    return off, metrics

# ---------------------------------------------------------------------
# CHANGED: main – same flow, but uses global dirs and saves outputs
# ---------------------------------------------------------------------
if __name__ == "__main__":
    os.makedirs(OUTPUTS_DIR, exist_ok=True)

    # 1) Train on base week (e.g., week 1)
    #base_week = 1
    #base_csv = os.path.join(TRAIN_DIR, f"input_2023_w{base_week:02d}.csv")
    base_csv = "input_data_clean.csv"
    print(f"Training on {base_csv} ...")
    df_in = pd.read_csv(base_csv)

    # train_and_evaluate is assumed to already print its own metrics
    # and return play_stats, which we use to summarize training performance
    model, off_df, play_stats = train_and_evaluate(df_in)

    # Build a row of metrics for the training split
    train_metrics = {
        #"dataset": f"week_{base_week:02d}",
        "split_type": "train_split",
        "top1_accuracy": float(play_stats["top1_correct"].mean()),
        "mean_true_prob": float(play_stats["true_target_prob"].mean()),
        "mean_pos_log_loss": float(play_stats["pos_log_loss"].mean()),
    }

    # 2) Evaluate on multiple weeks and accumulate predictions + metrics
    #weeks_to_test = [1, 2, 3, 4]   # include 1 so you can compare train_split vs full week1
    all_predictions = []
    metrics_rows = [train_metrics]

    #for wk in weeks_to_test:
     #   csv_path = os.path.join(TRAIN_DIR, f"input_2023_w{wk:02d}.csv")
      #  try:
       #     week_preds, week_metrics = evaluate_week_with_model(model, csv_path)
        #    week_metrics["dataset"] = f"week_{wk:02d}"
         #   week_metrics["split_type"] = "eval_week"
         #   metrics_rows.append(week_metrics)
          #  all_predictions.append(week_preds)
        #except FileNotFoundError:
         #   print(f"File not found: {csv_path} (skipping)")

    all_predictions = evaluate_week_with_model(model, base_csv)

    # 3) Save combined predictions (unchanged behavior)
    if all_predictions:
        all_preds_df = pd.concat(all_predictions, ignore_index=True)

        cols_to_save = [
            "game_id", "play_id", "nfl_id", "player_name",
            "is_target",           # may be NaN if no labels for that week
            "predicted_target",
            "target_prob",
        ]
        cols_to_save = [c for c in cols_to_save if c in all_preds_df.columns]

        out_path = os.path.join(OUTPUTS_DIR, "predicted_targets_by_play.csv")
        all_preds_df[cols_to_save].to_csv(out_path, index=False)
        print(f"\nSaved combined predictions to: {out_path}")
    else:
        print("No predictions generated; nothing to save.")

    # 4) Build and print a metrics grid for easy comparison
    if metrics_rows:
        metrics_df = pd.DataFrame(metrics_rows)

        # Order columns nicely
        col_order = [
            "dataset",
            "split_type",
            "top1_accuracy",
            "mean_true_prob",
            "mean_pos_log_loss",
        ]
        metrics_df = metrics_df[[c for c in col_order if c in metrics_df.columns]]

        print("\n=== Metrics Summary Grid ===")
        print(metrics_df.to_string(index=False, float_format="%.6f"))

        # Optionally save to CSV for tracking runs
        metrics_out = os.path.join(OUTPUTS_DIR, "metrics_summary.csv")
        metrics_df.to_csv(metrics_out, index=False)
        print(f"\nSaved metrics summary to: {metrics_out}")

### KNN

In [ ]:
# Load supplementary and weekly data
df1 = load_supplementary()
df2 = load_weekly_data()
# Create pass completed boolean
df1['pass_completed'] = df1['pass_result'].apply(lambda x: 1 if x=='C' else 0)
# Filter supplementary and weekly data sets
df1 = df1[["game_id", "play_id", "down", "yards_to_go", "yardline_number", "defenders_in_the_box",
            "dropback_distance", "pass_completed"]]
df2 = df2[["game_id", "play_id", "player_name", "player_role", "x", "y", "s", "a", "o"]]
# Set player role df
df_qb_wr = df2[df2['player_role'].isin(['Passer', 'Targeted Receiver'])]
# Use pivot table to widen data
qb_wr_wide = df_qb_wr.pivot_table(
        index=["game_id", "play_id"],
        columns="player_role",
        values=["x", "y", "o", "s", "a"],
        aggfunc="last"
    )
# Get column names
qb_wr_wide.columns = [f"{v}_{r}" for v, r in qb_wr_wide.columns]
qb_wr_wide = qb_wr_wide.reset_index()
# Get distance from qb to wr
qb_wr_wide["distance_qb_wr"] = np.sqrt(
        (qb_wr_wide["x_Passer"] - qb_wr_wide["x_Targeted Receiver"])**2 +
        (qb_wr_wide["y_Passer"] - qb_wr_wide["y_Targeted Receiver"])**2
    )
# Get orientation difference between qb and wr
qb_wr_wide["orientation_diff"] = np.abs(qb_wr_wide["o_Passer"] - qb_wr_wide["o_Targeted Receiver"])
# Rename cols. 
qb_wr_wide = qb_wr_wide.rename(columns={
        "s_Targeted Receiver": "wr_speed",
        "a_Targeted Receiver": "wr_accel"
    })
# Filter datasets
qb_wr_wide = qb_wr_wide[["game_id", "play_id", "distance_qb_wr", "orientation_diff", "wr_speed", "wr_accel"]]
df_qb_wr = df_qb_wr[["game_id", "play_id", "player_name", "player_role"]] 
# Merge datasets
model_df = df_qb_wr.merge(
        qb_wr_wide,
        on=["game_id", "play_id"],
        how="inner"
    )
# Remove duplicate plays
model_df = model_df.drop_duplicates(subset="play_id", keep="first")
# Join model data for weekly data with supplementary data
model_df = model_df.merge(
        df1,
        on=["game_id", "play_id"],
        how="inner"
    )
model_df = model_df.dropna() # Remove nas
model_df.to_csv('knn.csv', index=False) # Write csv
# Set target and response variables
y = model_df["pass_completed"]
X = model_df[[
    "distance_qb_wr",
    "orientation_diff",
    "dropback_distance",
    "wr_speed",
    "wr_accel"
]]
# Set train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y # Stratify to ensure equal proportion
)
# Set model pipeline with standard scaler and euclidean distance
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(weights="distance"))
])
# Set up grid search for number of neighbors
param_grid = {"knn__n_neighbors": range(1, 41, 2)}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="balanced_accuracy", n_jobs=-1)
grid.fit(X_train, y_train) # Fit model
# Build results dataframe
results_df = pd.DataFrame(grid.cv_results_)
# Store k and balanced accuracy
results_df["k"] = results_df["param_knn__n_neighbors"]
results_df["mean_score"] = results_df["mean_test_score"]
# Get best k and balanced accuracy
best_k = grid.best_params_["knn__n_neighbors"]
best_score = grid.best_score_
# Build model pipeline with best k
pipe2 = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k,
    weights="distance"))
])
# Fit model
pipe2.fit(X_train, y_train)
y_pred = pipe2.predict(X_test) # Predict on test set
# Get accuracy and balanced accuracy for model
acc = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)
# Print out accuracy and balanced accuracy
print(f"Accuracy: {acc:.3f}")
print(f"Balanced accuracy: {bal_acc:.3f}")

### TSNE, UMap with hyperparameter tuning

#### Sources

https://plotly.com/python/t-sne-and-umap-projections/ 

https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html 

https://umap-learn.readthedocs.io/en/latest/basic_usage.html 

In [ ]:
# data prepraation
positions = pre_snap["player_position"].values # assign labels 
roles = pre_snap["player_role"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# take subset of data HERE because t-sne and umap are slow

n_sample = min(5000, len(X_scaled)) 
sample_idx = np.random.choice(len(X_scaled), n_sample, replace=False)
X_sample = X_scaled[sample_idx]
positions_sample = positions[sample_idx] # assign labels
roles_sample = roles[sample_idx] # assign labels

# initialize and fit t-SNE and UMAP

 # reducing TNSE to 2D, the parameter being tuned
    # Need to initialize t-SNE with PCA here 
    # lets t-SNE choose an appropriate learning rate

tsne = TSNE(
    n_components=2, 
    init="pca", 
    learning_rate="auto", 
    random_state=42
).fit_transform(X_sample)

umap_result = umap.UMAP(
    n_components=2, 
    random_state=42
).fit_transform(X_sample)

In [ ]:
# establish data frame of results for both t-SNE and UMAP
tsne_df = pd.DataFrame({
    'tsne1': tsne[:, 0],
    'tsne2': tsne[:, 1],
    'position': positions_sample,
    'role': roles_sample
})

umap_df = pd.DataFrame({
    'umap1': umap_result[:, 0],
    'umap2': umap_result[:, 1],
    'position': positions_sample,
    'role': roles_sample
})

In [ ]:
tsne_df.head()

In [ ]:
umap_df.head()

In [ ]:
pca = PCA(n_components=2, random_state=42).fit_transform(X_sample)
# want to compare with t-SNE and UMAP, so will reduce to 2 PCs for consistency and interpretability

for emb, title in zip([pca, tsne, umap_result], ["PCA", "t-SNE", "UMAP"]): # looping through all models to plot them 
    plt.figure(figsize=(10, 8))
    uniq = np.unique(positions_sample) # grab unique positions 

    for pos in uniq:
        mask = positions_sample == pos
        plt.scatter(emb[mask, 0], emb[mask, 1], s=20, alpha=0.8, label = pos) # define clustering for each position 

    # plot formatting and such 
    plt.xlabel("Dim 1")
    plt.ylabel("Dim 2")

    plt.title(title)

    plt.legend(title="Position")
    plt.show()

In [ ]:
# Hyperparameter Tuning 
# Tuning perplexity for t-SNE

def scatter_by_position(emb, title):
    plt.figure(figsize=(10, 8))
    for pos in np.unique(positions_sample):
        m = positions_sample == pos # position selection here
        plt.scatter(emb[m, 0], emb[m, 1], s=18, label=pos)
    
    # plot formatting
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel("Dim 1", fontsize=12)
    plt.ylabel("Dim 2", fontsize=12)
    plt.legend(title = "Position")

    plt.show()

# Parameter to tune for t-SNE: perplexity
for perp in [5, 30, 50]:
    emb = TSNE(n_components=2, perplexity=perp, init="pca", learning_rate="auto", random_state=42).fit_transform(X_sample)

    # reducing TNSE to 2D, the parameter being tuned
    # Need to initialize t-SNE with PCA here 
    # lets t-SNE choose an appropriate learning rate

    scatter_by_position(emb, f"t-SNE (perplexity={perp})")

In [ ]:
# Hyperparamter Tuning: tuning n_neighbors & min_dist for umap

for nn in [5, 15, 50]: # testing various neighbor sizes
    for md in [0.0, 0.5]: # various tightness within the cluster itself 
        emb = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=md, random_state=42).fit_transform(X_sample)
        scatter_by_position(emb, f"UMAP (n_neighbors={nn}, min_dist={md})")

### Streamlit App